In [ ]:
import pandas as pd
import numpy as np
import json
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

In [ ]:
users = pd.read_csv('users.csv')
print("Users Data:")
display(users.head())
print("Shape:", users.shape)

In [ ]:
with open('sales.json', 'r') as f:
    sales_data = json.load(f)

sales = pd.DataFrame(sales_data)
print("Sales Data:")
display(sales.head())
print("Shape:", sales.shape)

In [ ]:
conn = sqlite3.connect(':memory:')

with open('inventory.sql', 'r') as f:
    sql_script = f.read()

conn.executescript(sql_script)

inventory = pd.read_sql_query("SELECT * FROM products", conn)
print("Inventory Data:")
display(inventory.head())
print("Shape:", inventory.shape)

In [ ]:
print("Users Info:")
users.info()
print("\nMissing Values in Users:")
print(users.isnull().sum())

print("\nSales Info:")
sales.info()
print("\nMissing Values in Sales:")
print(sales.isnull().sum())

print("\nInventory Info:")
inventory.info()
print("\nMissing Values in Inventory:")
print(inventory.isnull().sum())

In [ ]:
users['age'] = users['age'].fillna(users['age'].mean())
users['gender'] = users['gender'].fillna(users['gender'].mode()[0])

sales['amount'] = sales['amount'].fillna(sales['amount'].mean())

inventory['price'] = inventory['price'].fillna(inventory['price'].mean())

print("Missing values filled.")
print("Users missing:", users.isnull().sum().sum())
print("Sales missing:", sales.isnull().sum().sum())
print("Inventory missing:", inventory.isnull().sum().sum())

In [ ]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[column] = np.where(df[column] < lower, lower, df[column])
    df[column] = np.where(df[column] > upper, upper, df[column])
    return df

users = remove_outliers(users, 'age')
sales = remove_outliers(sales, 'amount')
inventory = remove_outliers(inventory, 'price')

print("Outliers handled.")

In [ ]:
users['registration_date'] = pd.to_datetime(users['registration_date'])
users['reg_year'] = users['registration_date'].dt.year
users['reg_month'] = users['registration_date'].dt.month
users['reg_day'] = users['registration_date'].dt.day
users = users.drop('registration_date', axis=1)

sales['date'] = pd.to_datetime(sales['date'])
sales['sale_year'] = sales['date'].dt.year
sales['sale_month'] = sales['date'].dt.month
sales['sale_day'] = sales['date'].dt.day
sales = sales.drop('date', axis=1)

print("Date columns converted.")

In [ ]:
le = LabelEncoder()
users['gender'] = le.fit_transform(users['gender'])
users['city'] = le.fit_transform(users['city'])

sales['payment_type'] = le.fit_transform(sales['payment_type'])

inventory['category'] = le.fit_transform(inventory['category'])

print("Text converted to numbers.")
display(users.head())

In [ ]:
merged = sales.merge(users, on='user_id', how='left')
merged = merged.merge(inventory, on='product_id', how='left')

print("Merged Data Shape:", merged.shape)
display(merged.head())

In [ ]:
merged['total_spend'] = merged['amount'] * merged['price']
merged['tax'] = merged['total_spend'] * 0.18
merged['final_amount'] = merged['total_spend'] + merged['tax']

print("New features created.")
display(merged[['total_spend', 'tax', 'final_amount']].head())

In [ ]:
scaler = StandardScaler()
scale_cols = ['age', 'amount', 'price', 'total_spend', 'final_amount']
merged[scale_cols] = scaler.fit_transform(merged[scale_cols])

print("Scaling done.")
display(merged[scale_cols].head())

In [ ]:
print("Final Dataset Shape:", merged.shape)
print("\nFinal Columns:")
print(merged.columns.tolist())

plt.figure(figsize=(8,5))
sns.heatmap(merged[['age', 'amount', 'price', 'total_spend', 'final_amount']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

merged.to_csv('final_cleaned_dataset.csv', index=False)
print("File saved as final_cleaned_dataset.csv")